# Join Measurements and Bookings Datasets
This notebook performs a left join between two parquet datasets (`measurements_non_labeled.parquet` and `bookings_non_labeled.parquet`) based on `serial_number_id` and `booking_id`. It outputs:
- Joined parquet file
- CSVs for unmatched measurements and bookings
- Timestamp comparison CSV
- Summary CSV with join statistics and metadata

 The datasets are large, so the join is done in batches for memory efficiency.

 ---

-## 1. Imports and File Paths Setup
Import required libraries and define all file paths and output locations.

In [ ]:
import json
import os
from datetime import datetime

import pandas as pd
import polars as pl
import pyarrow as pa
import pyarrow.parquet as pq
from tqdm import tqdm

# Define paths
measurements_file = r"M:\Universität\Master\Semester2\RealWorld_ML_Problems\Data\data_hella_single_line\output\measurements_non_labeled.parquet"
bookings_file = r"M:\Universität\Master\Semester2\RealWorld_ML_Problems\Data\data_hella_single_line\output\bookings_non_labeled.parquet"
output_dir = r"M:\Universität\Master\Semester2\RealWorld_ML_Problems\Data\data_hella_single_line\output"
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
result_file = os.path.join(output_dir, f"joined_measurements_bookings_{timestamp}.parquet")
unmatched_measurements_csv = os.path.join(output_dir, f"unmatched_measurements_{timestamp}.csv")
unmatched_bookings_csv = os.path.join(output_dir, f"unmatched_bookings_{timestamp}.csv")
summary_file = os.path.join(output_dir, f"join_summary_{timestamp}.csv")
timestamp_comparison_csv = os.path.join(output_dir, f"timestamp_comparison_{timestamp}.csv")

# Ensure output directory exists
os.makedirs(output_dir, exist_ok=True)

# Verify file existence
for file_path in [measurements_file, bookings_file]:
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Parquet file not found at {file_path}. Please check the file path or name.")

## 2. Count Total Rows in Each Parquet File
Define a helper function to count rows in parquet files and print the counts for both datasets.

In [ ]:
# Initialize batch size
batch_size = 2_000_000


# Function to count total rows
def count_parquet_rows(file_path, label="file"):
    total = pq.ParquetFile(file_path).metadata.num_rows
    print(f"Total rows in {label}: {total:,}")
    return total


# Count rows in both files
measurements_total_rows = count_parquet_rows(measurements_file, "measurements")
bookings_total_rows = count_parquet_rows(bookings_file, "bookings")

## 3. Define Function to Perform Left Join
The main function that:
- Reads both parquet files
- Performs batch-wise left join on `serial_number_id` and `booking_id`
- Tracks unmatched rows
- Writes the joined data to a parquet file
- Collects earliest/latest timestamps and other metadata
- Returns join statistics and data for output files

In [ ]:
def perform_left_join(measurements_path, bookings_path, result_file, unmatched_measurements_csv, unmatched_bookings_csv,
                      batch_size=10000):
    bookings_df, bookings_schema = load_and_prepare_bookings(bookings_path)
    measurements_parquet = pq.ParquetFile(measurements_path)
    measurements_schema = measurements_parquet.schema_arrow
    combined_schema = create_combined_schema(measurements_schema, bookings_schema)

    joined_writer = pq.ParquetWriter(result_file, combined_schema, compression='snappy')
    tracking = initialize_tracking()
    matched_booking_indices = set()

    with tqdm(total=measurements_parquet.metadata.num_rows, desc="Processing measurements for join",
              unit="rows") as pbar:
        row_counter = 0

        for batch in measurements_parquet.iter_batches(batch_size=batch_size, use_threads=True):
            df_batch = pl.from_arrow(batch)
            batch_indices = range(row_counter, row_counter + df_batch.height)

            update_timestamp_tracking(df_batch, 'created_at', batch_indices, tracking['measurements'])

            matched, unmatched, matched_indices, timestamp_data = join_and_process_batch(df_batch, bookings_df)
            matched_booking_indices.update(matched_indices)

            if not matched.is_empty():
                joined_writer.write_table(matched.to_arrow(), schema=combined_schema)
                tracking['joined_row_count'] += matched.height
                update_timestamp_tracking(matched, 'created_at', batch_indices, tracking['joined'])

            if not unmatched.is_empty():
                unmatched.write_csv(unmatched_measurements_csv, append=True,
                                    has_header=not tracking['unmatched_measurements_written'])
                tracking['unmatched_measurements_count'] += unmatched.height
                tracking['unmatched_measurements_written'] = True

            tracking['timestamp_comparison_data'].extend(timestamp_data)
            row_counter += df_batch.height
            pbar.update(df_batch.height)

            del df_batch, matched, unmatched  # help GC

    joined_writer.close()

    unmatched_bookings_df = bookings_df.filter(
        ~pl.struct(['serial_number_id', 'booking_id']).is_in(list(matched_booking_indices)))
    if unmatched_bookings_df.height > 0:
        unmatched_bookings_df.write_csv(unmatched_bookings_csv)

    tracking['unmatched_bookings_count'] = unmatched_bookings_df.height

    # Use Polars to find min/max for bookings
    full_bookings_df = pl.read_parquet(bookings_path)
    update_timestamp_tracking(full_bookings_df, 'book_stamp', range(full_bookings_df.height), tracking['bookings'])

    return build_result(tracking)

In [ ]:
def load_and_prepare_bookings(bookings_path):
    bookings_df = pl.read_parquet(bookings_path)
    rename_map = {col: f"{col}_b" for col in bookings_df.columns if col not in ['serial_number_id', 'booking_id']}
    bookings_df = bookings_df.rename(rename_map)
    bookings_df = bookings_df.with_columns([
        pl.col("serial_number_id").cast(pl.Utf8),
        pl.col("booking_id").cast(pl.Utf8),
    ])
    bookings_schema = pq.ParquetFile(bookings_path).schema_arrow
    return bookings_df, bookings_schema

In [ ]:
def create_combined_schema(measurements_schema, bookings_schema):
    combined_fields = measurements_schema
    for field in bookings_schema:
        if field.name not in ['serial_number_id', 'booking_id']:
            combined_fields = combined_fields.append(pa.field(f"{field.name}_b", field.type))
    return combined_fields

In [ ]:
def initialize_tracking():
    return {
        'joined_row_count': 0,
        'unmatched_measurements_count': 0,
        'unmatched_measurements_written': False,
        'unmatched_bookings_count': 0,
        'timestamp_comparison_data': [],
        'measurements': create_timestamp_tracker(),
        'bookings': create_timestamp_tracker(),
        'joined': create_timestamp_tracker()
    }

In [ ]:
def create_timestamp_tracker():
    return {'time': None, 'row': None, 'row_number': None}

In [ ]:
def update_timestamp_tracking(df: pl.DataFrame, time_col: str, row_indices, tracker):
    if time_col not in df.columns or df.is_empty():
        return

    batch_min = df[time_col].min()
    batch_max = df[time_col].max()

    if batch_min is not None:
        min_row_idx = df.select(pl.col(time_col)).arg_min().to_numpy()[0]
        min_row = df.row(min_row_idx)
        min_row_number = row_indices[min_row_idx]
        if tracker['time'] is None or batch_min < tracker['time']:
            tracker.update({'time': batch_min, 'row': dict(zip(df.columns, min_row)), 'row_number': min_row_number})

    if batch_max is not None:
        max_row_idx = df.select(pl.col(time_col)).arg_max().to_numpy()[0]
        max_row = df.row(max_row_idx)
        max_row_number = row_indices[max_row_idx]
        if tracker['time'] is None or batch_max > tracker['time']:
            tracker.update({'time': batch_max, 'row': dict(zip(df.columns, max_row)), 'row_number': max_row_number})

In [ ]:
def join_and_process_batch(df_batch, bookings_df):
    df_batch = df_batch.with_columns([
        pl.col("serial_number_id").cast(pl.Utf8),
        pl.col("booking_id").cast(pl.Utf8),
    ])

    joined = df_batch.join(bookings_df, on=['serial_number_id', 'booking_id'], how='left')

    matched = joined.filter(pl.col(f"{bookings_df.columns[-1]}").is_not_null())
    unmatched = joined.filter(pl.col(f"{bookings_df.columns[-1]}").is_null())

    matched_indices = set(zip(matched['serial_number_id'], matched['booking_id']))

    timestamp_data = []
    if 'created_at' in matched.columns and 'book_stamp_b' in matched.columns:
        timestamp_data = matched.select(['serial_number_id', 'booking_id', 'created_at', 'book_stamp_b']).to_dicts()

    return matched, unmatched, matched_indices, timestamp_data

In [ ]:
def build_result(tracking):
    return (
        tracking['joined_row_count'],
        tracking['unmatched_measurements_count'],
        tracking['unmatched_bookings_count'],
        tracking['measurements'],
        tracking['measurements'],  # earliest and latest are the same tracker; we stored both min/max in it
        tracking['bookings'],
        tracking['bookings'],
        tracking['joined'],
        tracking['joined'],
        tracking['timestamp_comparison_data']
    )

## 4. Run the Join and Collect Results
Call the join function and collect the output variables for further processing.

In [ ]:
joined_count, unmatched_measurements_count, unmatched_bookings_count, \
    earliest_measurements, latest_measurements, \
    earliest_bookings, latest_bookings, \
    earliest_joined, latest_joined, \
    timestamp_comparison_data = perform_left_join(
    measurements_file,
    bookings_file,
    result_file="joined_output.parquet",
    unmatched_measurements_csv="unmatched_measurements.csv",
    unmatched_bookings_csv="unmatched_bookings.csv",
    batch_size=10000  # Optional: default=10000
)

## 5. Preview Joined Result
Load and display the first 5 rows of the joined parquet file for a quick check.

In [ ]:
# Get 5 rows from result file
result_peek = pd.read_parquet(result_file, engine='pyarrow').head(5).to_dict('records')

## 6. Generate Timestamp Comparison CSV
Create a CSV file comparing `created_at` from measurements with `book_stamp` from bookings for matched rows.

In [ ]:
# Prepare timestamp comparison CSV using Polars
timestamp_comparison_df = pl.DataFrame(timestamp_comparison_data)

description = f"""\
# Script Purpose:
# Compares created_at from measurements and book_stamp from bookings for matched rows.
# Total Matched Rows: {len(timestamp_comparison_data)}
"""

with open(timestamp_comparison_csv, 'w', encoding='utf-8') as f:
    f.write(description + '\n')
    timestamp_comparison_df.write_csv(f, has_header=True)

## 7. Generate Summary CSV
Summarize the join operation statistics and earliest/latest row info from measurements, bookings, and joined datasets.

In [ ]:
def format_row_number(row_number, total_rows):
    return f"{row_number}/{total_rows}" if row_number is not None else None


def write_csv_with_description(description, df, filename):
    with open(filename, 'w', encoding='utf-8') as f:
        f.write(description)
        df.to_csv(f, index=False, lineterminator='\n')


# Calculate dataset peeks
result_peek = pd.read_parquet(result_file, engine='pyarrow').head(5).to_dict('records')

# Timestamp comparison CSV
timestamp_comparison_df = pd.DataFrame(timestamp_comparison_data)
description_text = f"""\
# Script Purpose:
# Performs a left join between measurements and bookings based on serial_number_id and booking_id.
# Adds _b suffix to booking columns.
# Outputs joined parquet, unmatched measurements, unmatched bookings, timestamp comparisons, and this summary.
# Join Results:
# Total Measurements Rows: {measurements_total_rows}
# Total Bookings Rows: {bookings_total_rows}
# Joined Rows: {joined_count}
# Unmatched Measurements Rows: {unmatched_measurements_count}
# Unmatched Bookings Rows: {unmatched_bookings_count}
# Output Files:
# Joined Result: {result_file}
# Unmatched Measurements CSV: {unmatched_measurements_csv}
# Unmatched Bookings CSV: {unmatched_bookings_csv}
# Timestamp Comparison CSV: {timestamp_comparison_csv}
# Summary CSV: {summary_file}
"""

write_csv_with_description(description_text, timestamp_comparison_df, timestamp_comparison_csv)

In [ ]:
# Prepare summary data
summary_metrics = [
    ("Total Measurements Rows", measurements_total_rows),
    ("Total Bookings Rows", bookings_total_rows),
    ("Joined Rows", joined_count),
    ("Unmatched Measurements Rows", unmatched_measurements_count),
    ("Unmatched Bookings Rows", unmatched_bookings_count),
    ("Earliest Measurements Time", earliest_measurements['time']),
    ("Earliest Measurements Row Number",
     format_row_number(earliest_measurements['row_number'], measurements_total_rows)),
    ("Latest Measurements Time", latest_measurements['time']),
    ("Latest Measurements Row Number", format_row_number(latest_measurements['row_number'], measurements_total_rows)),
    ("Earliest Bookings Time", earliest_bookings['time']),
    ("Earliest Bookings Row Number", format_row_number(earliest_bookings['row_number'], bookings_total_rows)),
    ("Latest Bookings Time", latest_bookings['time']),
    ("Latest Bookings Row Number", format_row_number(latest_bookings['row_number'], bookings_total_rows)),
    ("Earliest Joined Time", earliest_joined['time']),
    ("Earliest Joined Row Number", format_row_number(earliest_joined['row_number'], joined_count)),
    ("Latest Joined Time", latest_joined['time']),
    ("Latest Joined Row Number", format_row_number(latest_joined['row_number'], joined_count)),
    ("Earliest Measurements Row", json.dumps(earliest_measurements['row'], default=str)),
    ("Latest Measurements Row", json.dumps(latest_measurements['row'], default=str)),
    ("Earliest Bookings Row", json.dumps(earliest_bookings['row'], default=str)),
    ("Latest Bookings Row", json.dumps(latest_bookings['row'], default=str)),
    ("Earliest Joined Row", json.dumps(earliest_joined['row'], default=str)),
    ("Latest Joined Row", json.dumps(latest_joined['row'], default=str)),
    ("Result Dataset Peek (First 5 Rows)", json.dumps(result_peek, default=str))
]

summary_df = pd.DataFrame(summary_metrics, columns=["Metric", "Value"])
write_csv_with_description(description_text, summary_df, summary_file)

print(f"Processing complete. Results saved to:")
print(f"  ✅ Joined Result: {result_file}")
print(f"  ✅ Unmatched Measurements CSV: {unmatched_measurements_csv}")
print(f"  ✅ Unmatched Bookings CSV: {unmatched_bookings_csv}")
print(f"  ✅ Timestamp Comparison CSV: {timestamp_comparison_csv}")
print(f"  ✅ Summary CSV: {summary_file}")